# Autonomous English-to-Indic Video Dubbing Pipeline

This recipe demonstrates an autonomous, production-grade video dubbing system that translates and dubs English videos into Indian languages using **strictly Bodhan Indic AI models**:
- `indic-transcribe` — Speech to text with boundary snapping
- `indic-translate` — Neural machine translation with terminology and script control
- `indic-speak` — Natural text-to-speech with calibrated voice profiles

**Requirements:** Set `BODHAN_API_KEY` in `.env` (or `BODHAN_TRANSCRIBE_KEY`, `BODHAN_TRANSLATE_KEY`, `BODHAN_SPEAK_KEY`). Get keys at [console.bodhan.ai](https://console.bodhan.ai).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, json, requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ.get("BODHAN_API_KEY", os.environ.get("BODHAN_TRANSCRIBE_KEY", ""))
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

def bodhan(method: str, path: str, **kw):
    resp = requests.request(method, f"{BASE_URL}{path}", headers={**HEADERS, **kw.pop("headers", {})}, timeout=120, **kw)
    if not resp.ok:
        try:
            err = resp.json()["error"]
            raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')}")
        except (ValueError, KeyError):
            resp.raise_for_status()
    return resp

print("Ready to dub with Bodhan AI API!")

## Step 1 — Speech-to-Text with `indic-transcribe`

Extract 16kHz mono audio from source video and transcribe speech segments using `indic-transcribe`.

In [ ]:
# Example transcription call using indic-transcribe
audio_path = Path("sample_data/sample.wav")
if audio_path.exists():
    with open(audio_path, "rb") as f:
        transcription = bodhan(
            "POST", "/v1/audio/transcriptions",
            files={"file": ("audio.wav", f, "audio/wav")},
            data={"model": "indic-transcribe", "language": "en"}
        ).json()
    print("Transcribed text:", transcription.get("text", ""))
else:
    print("Ready: indic-transcribe endpoint configured.")

## Step 2 — Translation with `indic-translate`

Translate English sentence segments into the target Indic language (e.g. Hindi `hi`, Tamil `ta`).

In [ ]:
english_text = "Welcome to the autonomous video dubbing pipeline powered by Bodhan AI."
target_lang = "hi"

translation_response = bodhan(
    "POST", "/translate",
    json={"text": english_text, "target_language": target_lang}
).json()

indic_text = translation_response.get("translation", "")
print(f"Target ({target_lang}):", indic_text)

## Step 3 — Speech Synthesis with `indic-speak`

Synthesize natural, high-fidelity Indic speech using `indic-speak` and calibrated voice profiles.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
output_wav = "outputs/dubbed_segment.wav"

speech_response = bodhan(
    "POST", "/v1/audio/speech",
    json={
        "model": "indic-speak",
        "input": indic_text or "नमस्ते, बोधन एआई डबिंग पाइपलाइन में आपका स्वागत है।",
        "voice": "Kavya",
        "instructions": json.dumps({"lang": "hi"})
    }
)

with open(output_wav, "wb") as f:
    f.write(speech_response.content)

print(f"Synthesized audio saved to {output_wav}")

## Step 4 — Running the Complete Autonomous Pipeline

The full agentic dubbing pipeline handles VAD chunking, timeline synchronization, EBU R128 loudness normalization, and MP4 multiplexing:

In [ ]:
# Run autonomous dubbing CLI
# python -m src.bodhan_dub.cli --input path/to/video.mp4 --langs hi,ta --out ./out

## Wrap-up

You now have an end-to-end autonomous dubbing recipe powered entirely by Bodhan Indic AI models. Check `outputs/` and `report.json` for sync, stretch, and acoustic QA metrics.